## STEP 1) Data Preparation
In this step I prepared the CIFAR-10 dataset so it could be used to train the deep learning model. I applied transformations such as random cropping horizontal and vertical flipping rotation and normalisation to the images I then split the dataset into training, validation and testing sets and created data loaders to load the images in batches of 32

In [5]:
import torch
import torchvision.transforms as transforms
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader, random_split

In [21]:
# Define data augmentation and normalisation transformations
transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

In [22]:
# Load CIFAR-10 dataset
dataset = CIFAR10(root='./data', train=True, download=True, transform=transform)

100%|██████████| 170M/170M [17:23<00:00, 163kB/s] 


In [23]:
# Split dataset into training, validation, and testing sets
train_size = int(0.8 * len(dataset))
val_size = int(0.1 * len(dataset))
test_size = len(dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size])

In [24]:
# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

## STEP 2) Model Loading
In this step I loaded a pre trained ResNet18 model see this pre trained model has already been trained on a large dataset what means it can be adapted for a different task so I changed the final fully connected layer so that the model could classify the 10 different classes in the CIFAR-10 dataset

In [2]:
import torchvision.models as models

# Load a pre-trained ResNet18 model
model = models.resnet18(pretrained=True)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 164MB/s] 


In [8]:
# Modify the final layer to match the number of classes in CIFAR-10
num_ftrs = model.fc.in_features
model.fc = torch.nn.Linear(num_ftrs, 10)

## STEP 3) Training Script
In this step I prepared the model for training by checking if a GPU was available and moving the model to the selected device so I also created the CrossEntropyLoss function and SGD optimise the training loop then processes batches off images calculates the loss updates the model weights and records the loss for each epoch

In [12]:
import torch.optim as optim
import torch.nn as nn
from torch.cuda import is_available

# Check for GPU
device = torch.device("cuda:0" if is_available() else "cpu")
model = model.to(device)

# Loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

In [25]:
#Training loop
num_epochs = 2

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {running_loss/len(train_loader)}")

Epoch 1/2, Loss: 1.1739369038105012
Epoch 2/2, Loss: 0.8642523535013199


## STEP 4) Data Loading and Batching
In this step the data loaders created earlier are used to handle the training plus validation and testing data the data is loaded in batches rather than all at once what makes it easier for the model to process the dataset during training the training data is also shuffled to help the model learn from the data in a varied order

## STEP 5) Saving and Loading Checkpoints
In this step I saved the trained model using its state dictionary and stored it as model.pth I then loaded the checkpoint back into the model and changed the model into evaluation mode using model.eval(). This allows the trained model to be used later for inference without needing to train it again

In [26]:
# Save the model checkpoint
torch.save(model.state_dict(), 'model.pth')

# Load the model checkpoint
model.load_state_dict(torch.load('model.pth'))
model.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

## STEP 6) Inference with transformations
In this step I prepared an image for the trained model and used inference to make a prediction the image was resized centre cropped converted into a tensor and normalised before being given a batch dimension the model was then placed into evaluation mode and used to predict one off the CIFAR-10 classes

In [33]:
from PIL import Image
import requests
from io import BytesIO

# Load an example image
# a few examples to test inference could be
# https://cdn.britannica.com/79/233277-050-6B0411D7/German-Shepherd-dog-Alsatian.jpg
# https://static1.simplecityimages.com/wordpress/wp-content/uploads/2023/08/emirates-a380-with-new-livery-by-vincenzo-pace-from-sf.jpg
# https://cdn.mos.cms.futurecdn.net/39CUYMP8VqHA7GvUhBN-1200-80.jpg
# https://i.natgeofe.com/n/548467d8-c5f1-4551-9f58-6817a8d2c45e/NationalGeographic_5272187_square.jpg

url = "https://picsum.photos/400/300"
response = requests.get(url)
img = Image.open(BytesIO(response.content))

In [34]:
# Apply the same transformations as used in training

transform_inference = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

img_t = transform_inference(img)
img_t = img_t.unsqueeze(0).to(device)  # Add batch dimension and move to GPU

In [35]:
# Inference

model.eval()

with torch.no_grad():
    output = model(img_t)
    _, predicted = torch.max(output, 1)
    print(f'Predicted class: {predicted.item()}')

Predicted class: 4
